#  Laboratorio de LLMs: Fine-tuning y Transfer Learning

**Modelo base:** `bert-base-multilingual-cased` · **Dataset:** `dair-ai/emotion` (6 categorías) · **Tarea:** clasificación de texto

**Idea central:** en lugar de entrenar un modelo desde cero, reutilizamos el conocimiento lingüístico que BERT ya adquirió en su preentrenamiento (*transfer learning*). Congelamos su cuerpo y entrenamos únicamente una pequeña cabeza de clasificación nueva.

>  En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*, y luego *Ejecutar todas*.

##  Fase 1: Configuración del entorno y modelo

**Paso 1.1 – Instalación.** `transformers` aporta el modelo y el `Trainer`; `datasets` descarga y procesa los datos; `accelerate` gestiona el uso de GPU/CPU durante el entrenamiento; `evaluate` calcula métricas.

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

**Paso 1.2 – Carga del tokenizador.** El tokenizador convierte texto en *input_ids* (números) usando el mismo vocabulario con el que se preentrenó BERT. Debe corresponder exactamente al modelo elegido, por eso se carga con el mismo nombre.

In [ ]:
import numpy as np, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import load_dataset
import evaluate

MODEL_NAME = "bert-base-multilingual-cased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verificación: el tokenizador funciona y se carga sin errores
ejemplo = tokenizer("Hola mundo, esto es una prueba")
print("input_ids:", ejemplo["input_ids"])
print("tokens   :", tokenizer.convert_ids_to_tokens(ejemplo["input_ids"]))
print(" Tokenizador cargado correctamente | tamaño de vocabulario:", tokenizer.vocab_size)

Dispositivo: cuda
input_ids: [101, 20220, 10330, 13137, 117, 18278, 10196, 10153, 34303, 102]
tokens   : ['[CLS]', 'Ho', '##la', 'mundo', ',', 'esto', 'es', 'una', 'prueba', '[SEP]']
 Tokenizador cargado correctamente | tamaño de vocabulario: 119547


##  Fase 2: Preparación del dataset

**Paso 2.1 – Carga.** Usamos `dair-ai/emotion`: textos cortos etiquetados con 6 emociones (tristeza, alegría, amor, ira, miedo, sorpresa). Cumple el requisito de **mínimo 2 categorías**.

**Paso 2.2 – Tokenización.** Se aplica el tokenizador a cada texto con `truncation=True` (recorta textos largos) y `padding="max_length"` (rellena los cortos) para que todos tengan 64 tokens y puedan agruparse en lotes (*batches*).

In [ ]:
dataset = load_dataset("dair-ai/emotion")
label_names = dataset["train"].features["label"].names
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in id2label.items()}
print("Categorías:", label_names, "| total:", len(label_names))
print("Ejemplo:", dataset["train"][0])

train_ds = dataset["train"]
test_ds  = dataset["test"]

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_tok = train_ds.map(tokenize, batched=True)
test_tok  = test_ds.map(tokenize, batched=True)
print(train_tok)

Categorías: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'] | total: 6
Ejemplo: {'text': 'i didnt feel humiliated', 'label': 0}
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16000
})


##  Fase 3: Implementación de Transfer Learning

**Paso 3.1 – Carga del modelo.** `AutoModelForSequenceClassification` carga el cuerpo de BERT con sus pesos preentrenados y le añade una **cabeza de clasificación nueva** (capa lineal de 768 → 6) con pesos aleatorios. El aviso *"Some weights ... newly initialized"* es normal y esperado: indica que esa cabeza aún no está entrenada.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names), id2label=id2label, label2id=label2id
)
print(" Modelo cargado correctamente:", model.__class__.__name__)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 Modelo cargado correctamente: BertForSequenceClassification


**Paso 3.2 –  Paso crítico: congelado de capas.** Al poner `requires_grad = False` en cada parámetro de `model.bert` (embeddings + 12 capas transformer + pooler), PyTorch deja de calcular gradientes para esos pesos y el optimizador no los modifica. Resultado: se conserva el conocimiento previo del LLM, se reduce el cómputo/memoria y se evita el sobreajuste. Solo la cabeza `classifier` seguirá aprendiendo.

In [ ]:
# Congelar TODAS las capas base de BERT
for param in model.bert.parameters():
    param.requires_grad = False

# --- Verificaciones explícitas ---
assert all(not p.requires_grad for p in model.bert.parameters()), "Error: hay capas base sin congelar"
assert all(p.requires_grad for p in model.classifier.parameters()), "Error: la cabeza debe ser entrenable"

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable
print(f"Parámetros totales     : {total:,}")
print(f"Parámetros CONGELADOS  : {frozen:,} ({100*frozen/total:.3f}%)")
print(f"Parámetros ENTRENABLES : {trainable:,} ({100*trainable/total:.4f}%)\n")
print("Capas entrenables:")
for name, p in model.named_parameters():
    if p.requires_grad:
        print("  ", name, tuple(p.shape))
print("\n requires_grad = False aplicado correctamente a las capas base")

Parámetros totales     : 177,858,054
Parámetros CONGELADOS  : 177,853,440 (99.997%)
Parámetros ENTRENABLES : 4,614 (0.0026%)

Capas entrenables:
   classifier.weight (6, 768)
   classifier.bias (6,)

 requires_grad = False aplicado correctamente a las capas base


##  Fase 4: Entrenamiento y evaluación

**Paso 4.1 – `TrainingArguments`.** Define los hiperparámetros: `num_train_epochs` (pasadas completas por el dataset), `per_device_train_batch_size` (ejemplos por lote), y `learning_rate`. Usamos una tasa de aprendizaje alta (`1e-3`) porque solo se entrena una capa pequeña desde cero; en fine-tuning completo se usaría ~`2e-5`.

**Paso 4.2 – `Trainer`.** Encapsula el ciclo de entrenamiento (forward, pérdida, backward, actualización) y la evaluación al final de cada epoch con la métrica *accuracy*.

In [ ]:
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy.compute(predictions=np.argmax(logits, axis=-1), references=labels)

training_args = TrainingArguments(
    output_dir="./resultados",
    num_train_epochs=5,                 # epochs
    per_device_train_batch_size=32,     # batch size
    per_device_eval_batch_size=64,
    learning_rate=1e-3,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
resultado = trainer.evaluate()
print("\n Resultado final en test:", resultado)

Epoch,Training Loss,Validation Loss,Accuracy
1,1.534265,1.504329,0.393000
2,1.517432,1.467158,0.421000
3,1.503679,1.458926,0.427000
4,1.468382,1.462619,0.418000
5,1.490972,1.448244,0.434000


Training Loss,Validation Loss,Epoch,Accuracy
1.490972,1.448244,5,0.434000



 Resultado final en test: {'eval_loss': 1.4482439756393433, 'eval_accuracy': 0.434}


**Paso 4.3 – Verificación de que las capas base no cambiaron.** Prueba de que el congelado funcionó: tras entrenar, los pesos de BERT siguen siendo los preentrenados.

In [ ]:
base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_names))
iguales = all(torch.equal(a.cpu(), b.cpu()) for a, b in zip(base.bert.parameters(), model.bert.parameters()))
print("¿Pesos de BERT idénticos a los originales tras entrenar?", iguales)
del base

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


¿Pesos de BERT idénticos a los originales tras entrenar? True


**Paso 4.4 – Inferencia de prueba.** Tokenizamos una frase nueva, obtenemos los *logits* del modelo y aplicamos **softmax** para convertirlos en probabilidades (suman 1). La categoría ganadora es la de mayor probabilidad, y esa probabilidad es el **score de confianza**.

In [ ]:
def clasificar(frase):
    model.eval()
    inputs = tokenizer(frase, return_tensors="pt", truncation=True, max_length=64).to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    ganadora = int(probs.argmax())
    print(f"\nFrase: {frase}")
    for i, p in sorted(enumerate(probs.tolist()), key=lambda x: -x[1]):
        print(f"  {id2label[i]:>9}: {p:.4f}  " + "" * int(p * 30))
    print(f"  >>> Categoría ganadora: {id2label[ganadora].upper()} | score de confianza: {probs[ganadora]:.4f} ({probs[ganadora]*100:.2f}%)")

# Frase nueva principal (puedes cambiarla)
clasificar("I am so happy today, everything is going great!")

# Pruebas adicionales
clasificar("I am furious, this is completely unacceptable!")
clasificar("I feel so sad and alone since she left.")
clasificar("I am terrified, I heard a strange noise in the house.")


Frase: I am so happy today, everything is going great!
        joy: 0.6626  
    sadness: 0.1518  
       love: 0.0846  
      anger: 0.0497  
   surprise: 0.0268  
       fear: 0.0245  
  >>> Categoría ganadora: JOY | score de confianza: 0.6626 (66.26%)

Frase: I am furious, this is completely unacceptable!
      anger: 0.2799  
    sadness: 0.2769  
        joy: 0.2187  
       fear: 0.1593  
   surprise: 0.0355  
       love: 0.0297  
  >>> Categoría ganadora: ANGER | score de confianza: 0.2799 (27.99%)

Frase: I feel so sad and alone since she left.
    sadness: 0.3479  
       fear: 0.2162  
      anger: 0.1876  
        joy: 0.1431  
       love: 0.0634  
   surprise: 0.0417  
  >>> Categoría ganadora: SADNESS | score de confianza: 0.3479 (34.79%)

Frase: I am terrified, I heard a strange noise in the house.
    sadness: 0.2958  
       fear: 0.2223  
      anger: 0.2091  
        joy: 0.1697  
   surprise: 0.0540  
       love: 0.0491  
  >>> Categoría ganadora: SADNESS | score